In [ ]:
GROQ_API_KEY="gsk_QU5vSKBM6VL8S5gaF6urWGdyb3FYttEDq28VBctplWeTpbib"
GEMINI_API_KEY="AQ.Ab8RN6LInw14KvrG7keh01NeQOK9aB5P_1tQ9DPt_"


##### 3q6b

##### 92Vf5M_Qw

In [5]:
import os
import json
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma


C:\Users\pc\AppData\Local\Temp\ipykernel_20304\2211787989.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [6]:
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')

In [7]:
JSON_FILE = "data.json"
VECTOR_DB = "./vectordb"

In [8]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
LLM_MODEL = "gpt-5"

In [7]:
metadatas =({"origin_process_id": "a",
            "app_name":"b", 
            

           })

In [8]:
type(metadatas)

dict

In [9]:
def load_json_documents(file_path):

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    documents = []

    for item in data:

        metadatas = ({"origin_process_id": item["origin_process_id"],
            "app_name":item['app_name'], 
            'step_name' :item['step_name'],
            'step_description':item['step_description']

           })
           
           
        

        documents.append(
            Document(
                page_content=json.dumps(item, indent=2),
                metadata=metadatas
            )
        )

    return documents

In [10]:
docs = load_json_documents('./data.json')

In [11]:
docs

[Document(metadata={'origin_process_id': 'custom_margins_20260723_102829', 'app_name': 'MS Word', 'step_name': 'msword_layout_margins', 'step_description': 'Click on the Margins drop-down menu in the Layout ribbon tab to expand page margin presets and custom options.'}, page_content='{\n  "step_id": "custom_margins_20260723_102829_step_001",\n  "origin_process_id": "custom_margins_20260723_102829",\n  "app_name": "MS Word",\n  "step_name": "msword_layout_margins",\n  "step_description": "Click on the Margins drop-down menu in the Layout ribbon tab to expand page margin presets and custom options.",\n  "action_type": "click",\n  "button": "Button.left",\n  "coordinates": {\n    "absolute_x": 57,\n    "absolute_y": 406\n  },\n  "relative_position_pct": {\n    "x_distance_from_left_pct": 2.97,\n    "y_distance_from_top_pct": 33.83,\n    "x_distance_from_right_pct": 97.03,\n    "y_distance_from_bottom_pct": 66.17\n  },\n  "visual_assets": {\n    "full_screenshot": "database/steps_repositor

In [11]:
from langchain_community.vectorstores import FAISS
def build_vector_database():

    print("Loading JSON...")

    documents = load_json_documents(JSON_FILE)

    print(f"Loaded {len(documents)} JSON records")

    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    vectorstore = FAISS.from_documents(documents,embeddings)
    vectorstore.save_local('Faiss_binni')
    print(f"index saved to Faiss_binni")
    print("Vector Database Created Successfully")

    return vectorstore

In [12]:
build_vector_database()

Loading JSON...
Loaded 11 JSON records
index saved to Faiss_binni
Vector Database Created Successfully


In [13]:
INDEX_DIR = 'Faiss_binni'

In [14]:
def load_vector_database():


    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    db = FAISS.load_local(INDEX_DIR, embeddings, allow_dangerous_deserialization=True)

    return db


In [15]:
def retrieve_documents(question, db, k=8):

    retriever = db.as_retriever(
        search_kwargs={"k": k}
    )

    docs = retriever.invoke(question)

    return docs

In [16]:
db = load_vector_database() 

In [17]:
res = retrieve_documents('insert page break',db)

In [18]:
from flashrank import Ranker,RerankRequest

In [19]:
_ranker = None


def _get_ranker() -> Ranker:
    """
    Initializes the FlashRank engine lazily. 
    FlashRank uses a local ONNX model (ms-marco-MiniLM-L-6-v2) for ultra-fast reranking.
    """
    global _ranker
    if _ranker is None:
        try:
            # We use a specific cache directory to avoid permission issues in production
            _ranker = Ranker(cache_dir="/tmp/flashrank")
        except Exception:
            _ranker = Ranker()
    return _ranker

In [25]:
def rerank_documents(query: str, documents: list[str], top_n: int = 8) -> list[str]:
    if not documents:
        return []

    try:
        ranker = _get_ranker()
        
        # FlashRank expects a list of dictionaries with 'id' and 'text'
        passages = [
            {"id": i, "text": doc}
            for i, doc in enumerate(documents)
        ]

        request = RerankRequest(query=query, passages=passages)
        results = ranker.rerank(request)
        
        # Results are returned sorted by highest semantic score first
        reranked_docs = []
        for res in results[:top_n]:
            reranked_docs.append(res['text'])

        top_score = results[0]['score'] if results else 'N/A'
        
        
        return reranked_docs

    except Exception as e:
        return documents[:top_n]

In [26]:
re_docs = rerank_documents('set custom margin to 1.5',res)

In [27]:
re_docs

[Document(id='d40a384f-84ac-4efc-ae45-e323aafbf96d', metadata={'origin_process_id': 'custom_margins_20260723_102829', 'app_name': 'MS Word', 'step_name': 'msword_bottom_margin', 'step_description': 'Type the Bottom Margin value into the Bottom margin input field.'}, page_content='{\n  "step_id": "custom_margins_20260723_102829_step_004",\n  "origin_process_id": "custom_margins_20260723_102829",\n  "app_name": "MS Word",\n  "step_name": "msword_bottom_margin",\n  "step_description": "Type the Bottom Margin value into the Bottom margin input field.",\n  "action_type": "type",\n  "text": "<bottom_margin>",\n  "coordinates": {\n    "absolute_x": 522,\n    "absolute_y": 270\n  },\n  "relative_position_pct": {\n    "x_distance_from_left_pct": 27.19,\n    "y_distance_from_top_pct": 22.5,\n    "x_distance_from_right_pct": 72.81,\n    "y_distance_from_bottom_pct": 77.5\n  },\n  "visual_assets": {\n    "full_screenshot": "database/steps_repository/images/custom_margins_20260723_102829_step_004_f

In [20]:
for r in res : 
    print("-------------------------------------------------")
    print(r)
    print("-------------------------------------------------")

-------------------------------------------------
page_content='{
  "step_id": "custom_margins_20260723_102829_step_005",
  "origin_process_id": "custom_margins_20260723_102829",
  "app_name": "MS Word",
  "step_name": "msword_left_margin",
  "step_description": "Type the Left Margin value into the Left margin input field.",
  "action_type": "type",
  "text": "<left_margin>",
  "coordinates": {
    "absolute_x": 522,
    "absolute_y": 305
  },
  "relative_position_pct": {
    "x_distance_from_left_pct": 27.19,
    "y_distance_from_top_pct": 25.42,
    "x_distance_from_right_pct": 72.81,
    "y_distance_from_bottom_pct": 74.58
  },
  "visual_assets": {
    "full_screenshot": "database/steps_repository/images/custom_margins_20260723_102829_step_005_full.png",
    "crop_element": "database/steps_repository/images/custom_margins_20260723_102829_step_005_crop.png",
    "crop_bounding_box": [
      470,
      280,
      610,
      330
    ]
  },
  "timestamp": "2026-07-23T10:28:40.522871"
}'

In [41]:
from langchain_groq import ChatGroq
def ask_llm(question, docs):

    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    prompt = f"""
You are an intelligent RAG assistant.

Below is the retrieved JSON data.

{context}

User Question:
{question}

Instructions:

1. Answer ONLY from the retrieved JSON.
2. Do not make up information.


3. Return ONLY valid JSON.
4. Do NOT return markdown.
"""

    response = llm.invoke(prompt)

    return response.content

# 3. If the answer is unavailable, return

# {{
#     "error":"Information not found"
# }}

In [42]:
question = "set custom margin to 1.5"
ans = ask_llm(question,docs)

In [43]:
print(ans)

{"step_id":"custom_margins_20260723_102829_step_001","origin_process_id":"custom_margins_20260723_102829","app_name":"MS Word","step_name":"msword_layout_margins","step_description":"Click on the Margins drop-down menu in the Layout ribbon tab to expand page margin presets and custom options.","action_type":"click","button":"Button.left","coordinates":{"absolute_x":57,"absolute_y":406},"relative_position_pct":{"x_distance_from_left_pct":2.97,"y_distance_from_top_pct":33.83,"x_distance_from_right_pct":97.03,"y_distance_from_bottom_pct":66.17},"visual_assets":{"full_screenshot":"database/steps_repository/images/custom_margins_20260723_102829_step_001_full.png","crop_element":"database/steps_repository/images/custom_margins_20260723_102829_step_001_crop.png","crop_bounding_box":[0,331,132,481]},"timestamp":"2026-07-23T10:28:31.772965"}


In [44]:
PROCESS_SCHEMA = {
    "process_id": "",
    "process_name": "",
    "app_name": "",
    "user_intent": "",
    "description": "",
    "recorded_at": "",
    "screen_dimensions": {
        "width_px": 0,
        "height_px": 0
    },
    "total_steps": 0,
    "is_parameterized": False,
    "required_parameters": [],
    "parameter_bindings": {},
    "process_sequence": []
}

In [45]:

db = FAISS.load_local(
    INDEX_DIR,
    embeddings,
    allow_dangerous_deserialization=True
)


In [46]:
retriever = db.as_retriever(
    search_kwargs={"k":5}
)

In [52]:
def build_context(user_query):

    docs = retriever.invoke(user_query)

    step_jsons = []

    for doc in docs:
        step_jsons.append(json.loads(doc.page_content))

    return step_jsons

In [53]:
build_context("insert a page break")

[{'step_id': 'page_orientation_20260728_001_step_001',
  'origin_process_id': 'page_orientation_20260728_001',
  'app_name': 'MS Word',
  'step_name': 'msword_page_orientation',
  'step_description': 'Click the Orientation button in the Layout ribbon to change the page orientation.',
  'action_type': 'click',
  'button': 'Button.left',
  'coordinates': {'absolute_x': 460, 'absolute_y': 89},
  'relative_position_pct': {'x_distance_from_left_pct': 23.96,
   'y_distance_from_top_pct': 7.12,
   'x_distance_from_right_pct': 76.04,
   'y_distance_from_bottom_pct': 92.88},
  'visual_assets': {'full_screenshot': 'database/steps_repository/images/page_orientation_20260728_001_step_001_full.png',
   'crop_element': 'database/steps_repository/images/page_orientation_20260728_001_step_001_crop.png',
   'crop_bounding_box': [430, 60, 505, 115]},
  'timestamp': '2026-07-28T10:27:31.874521'},
 {'step_id': 'insert_table_20260728_001_step_001',
  'origin_process_id': 'insert_table_20260728_001',
  'app

In [48]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

In [49]:
def generate_process_json(user_query):

    steps = build_context(user_query)

    prompt = f"""
You are an RPA Process Generator.

Below are individual automation step JSONs.

{json.dumps(steps, indent=4)}

Your task is to generate ONE process JSON.

Rules

1. Infer process_name.
2. Infer user_intent.
3. Infer description.
4. Copy app_name.
5. total_steps = number of retrieved steps.
6. process_sequence = list of step_ids.
7. If parameters are needed, populate required_parameters.
8. Otherwise keep them empty.
9. Return ONLY JSON.

Output Format

{{
    "process_id":"",
    "process_name":"",
    "app_name":"",
    "user_intent":"",
    "description":"",
    "recorded_at":"",
    "screen_dimensions":{{
        "width_px":1920,
        "height_px":1200
    }},
    "total_steps":0,
    "is_parameterized":false,
    "required_parameters":[],
    "parameter_bindings":{{}},
    "process_sequence":[]
}}
"""

    response = llm.invoke(prompt)

    return response.content


In [50]:
ans = generate_process_json('insert a page break ')

KeyError: 'json'